In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
path = '/content/drive/MyDrive/CSS2/A DENTAL INTRAORAL IMAGE DATASET OF GINGIVITIS FOR IMAGE CAPTIONING/Dataset/Training/Images'

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

## Data Preprocessing

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define transformation

# Add augmentation for training
train_transformation = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=0, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

transformation = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

### Define Dataset Class

In [ ]:
# Define dataset class to apply preprocessing and handle masks
class GingivitisDataset(Dataset):
  def __init__(self, image_folder, mask_folder, transform=None):
    self.image_folder = Path(image_folder)
    self.mask_folder = Path(mask_folder)
    self.transform = transform

    # Get all images that have a matching mask
    all_images = sorted(self.image_folder.glob("*.jpg"))
    self.image_paths = [p for p in all_images if (self.mask_folder / f"{p.stem}.png").exists()]

    print(f"Found {len(self.image_paths)}/{len(all_images)} images with matching masks in {image_folder}")

    # Define a simple colour-to-class mapping
    self.colour_to_label = {
      (0, 255, 0):    0,    # healthy
      (255, 255, 0): 1,    # mild
      (255, 165, 0):  2,    # moderate
      (255, 0, 0):  3,    # severe
      (139, 0, 0):    4,    # very severe
      (0, 0, 0):      255,  # background   - ignore during training
    }

  # Total number of images
  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    # Load image
    img_path = self.image_paths[idx]
    image_name = img_path.name

    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Load mask
    mask_path = self.mask_folder / f"{img_path.stem}.png"
    mask = cv2.imread(str(mask_path))

    # Check missing masks
    if mask is None:
      raise FileNotFoundError(f"Mask file not found or could not be loaded for image: {img_path}. Expected mask path: {mask_path}")

    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    # Apply transform
    if self.transform:
      augmented = self.transform(image=image, mask=mask)
      image = augmented["image"]
      mask = augmented["mask"]

    if isinstance(mask, torch.Tensor):
      mask = mask.cpu().numpy()

    mask = mask.astype(np.uint8)

    # Convert colours to labels
    # np.full defaults to 255 so unrecognised colours are ignored
    mask_label = np.full((mask.shape[0], mask.shape[1]), 255, dtype=np.uint8)

    tol = 5

    for colour, label in self.colour_to_label.items():
      target = np.array(colour)
      matches = np.all(np.abs(mask - target) <= tol, axis=-1) # add tolerance matching
      mask_label[(matches) & (mask_label == 255)] = label

    mask_label = torch.from_numpy(mask_label).long()

    return image, mask_label, image_name

### Create dataset objects

In [ ]:
from torch.utils.data import Subset
import random

# Base path for the dataset on Google Drive
BASE_PATH = "/content/drive/MyDrive/CSS2/A DENTAL INTRAORAL IMAGE DATASET OF GINGIVITIS FOR IMAGE CAPTIONING/Dataset"

train_dataset = GingivitisDataset(
    image_folder=f"{BASE_PATH}/Training/Images",
    mask_folder=f"{BASE_PATH}/Training/Masks",
    transform=train_transformation
)

val_dataset = GingivitisDataset(
    image_folder=f"{BASE_PATH}/Validation/Images",
    mask_folder=f"{BASE_PATH}/Validation/Masks",
    transform=transformation
)

test_dataset = GingivitisDataset(
    image_folder=f"{BASE_PATH}/Test/Images",
    mask_folder=f"{BASE_PATH}/Test/Masks",
    transform=transformation
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Found 732/732 images with matching masks in /content/drive/MyDrive/CSS2/A DENTAL INTRAORAL IMAGE DATASET OF GINGIVITIS FOR IMAGE CAPTIONING/Dataset/Training/Images
Found 182/182 images with matching masks in /content/drive/MyDrive/CSS2/A DENTAL INTRAORAL IMAGE DATASET OF GINGIVITIS FOR IMAGE CAPTIONING/Dataset/Validation/Images
Found 182/182 images with matching masks in /content/drive/MyDrive/CSS2/A DENTAL INTRAORAL IMAGE DATASET OF GINGIVITIS FOR IMAGE CAPTIONING/Dataset/Test/Images
Train: 732, Val: 182, Test: 182


## Create DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

## Load DeepLabV3+

DeepLabV3+ uses atrous (dilated) convolutions and an encoder-decoder structure with Atrous Spatial Pyramid Pooling (ASPP) to capture multi-scale context. The decoder refines segmentation boundaries by combining low-level and high-level features. Using ResNet50 as the backbone for a good balance of accuracy and efficiency.

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=5
)

model = model.to(device) # using GPU

In [ ]:
# Loss function and optimizer
criterion = torch.nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Training

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
  # Training loop
  model.train()
  total_loss = 0
  train_batches = 0

  for i, (images, masks, names) in enumerate(train_loader):
    images = images.to(device)   # using GPU
    masks = masks.to(device)

    # Skip batch if there are no valid pixels in the mask for loss calculation
    if not (masks != 255).any():
      prinmodel = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=5
)

model = model.to(device) # using GPUt(f"Training Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255)")
      continue

    outputs = model(images) # predict
    loss = criterion(outputs, masks) # calculate loss

    optimizer.zero_grad() # clear old gradients from previous steps
    loss.backward() # backward pass to calculate the gradients for all parameters
    optimizer.step() # update model

    total_loss += loss.item()
    train_batches += 1

  # Validation loop
  model.eval() # set model to evaluation mode
  val_loss = 0
  val_batches = 0

  with torch.no_grad():
    for i, (images, masks, names) in enumerate(val_loader):
      images = images.to(device)
      masks = masks.to(device)

      # Skip batch if there are no valid pixels in the mask for loss calculation
      if not (masks != 255).any():
        print(f"Validation Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255).")
        continue # Skip this batch

      outputs = model(images) # predict
      loss = criterion(outputs, masks) # calculate loss
      val_loss += loss.item()
      val_batches += 1

  train_loss = total_loss / train_batches if train_batches > 0 else 0
  val_loss = val_loss / val_batches if val_batches > 0 else 0

  print(f"Epoch {epoch+1}")
  print(f"Train loss: {train_loss:.2f}")
  print(f"Val loss: {val_loss:.2f}")

Epoch 1
Train loss: 1.41
Val loss: 1.38
Epoch 2
Train loss: 1.40
Val loss: 1.37
Epoch 3
Train loss: 1.39
Val loss: 1.37
Epoch 4
Train loss: 1.38
Val loss: 1.36
Epoch 5
Train loss: 1.39
Val loss: 1.38
Epoch 6
Train loss: 1.37
Val loss: 1.38
Epoch 7
Train loss: 1.38
Val loss: 1.39
Epoch 8
Train loss: 1.38
Val loss: 1.37
Epoch 9
Train loss: 1.37
Val loss: 1.40
Epoch 10
Train loss: 1.36
Val loss: 1.39


## Testing

IoU checks how much the prediction overlaps with the ground truth .  
Dice measures similarity between prediction and ground truth but gives more weight to overlapping pixels.  
Calculating the mean for these two to show one score overall for them.

In [ ]:
# Custom function cuz no built-in function for checking segmentation accuracy and even with some libraries, gotta manually remove 255 and reshape tensors
# Added IoU and Dice Metrics
def evaluate(model, dataloader, device, num_classes=5):
  model.eval() # set model to evaluation mode

  total_correct = 0
  total_pixels = 0

  iou_list = []
  dice_list = []

  with torch.no_grad():
    for images, masks, names in dataloader:
      images = images.to(device)
      masks = masks.to(device)

      outputs = model(images) # predict
      preds = torch.argmax(outputs, dim=1) # get predicted class for each label

      valid = masks != 255  # ignore background

      # Accuracy calculated on pixels
      correct = (preds == masks) & valid # number of correct predictions
      total_correct += correct.sum().item()
      total_pixels += valid.sum().item()

      # IoU and Dice
      for cls in range(num_classes):
        pred_cls = (preds == cls) # predicted pixels for current class
        mask_cls = (masks == cls) # true pixels for current class

        # ignore invalid pixels
        pred_cls = pred_cls & valid
        mask_cls = mask_cls & valid

        intersection = (pred_cls & mask_cls).sum().item() # pixels correctly predicted for this class
        union = (pred_cls | mask_cls).sum().item() # total pixels in prediction or ground truth

        if union > 0:
          iou = intersection / union # IoU formula
          dice = (2 * intersection) / (pred_cls.sum().item() + mask_cls.sum().item() + 1e-6) # dice formula

          iou_list.append(iou)
          dice_list.append(dice)

  accuracy = total_correct / total_pixels
  mean_iou = sum(iou_list) / len(iou_list)
  mean_dice = sum(dice_list) / len(dice_list)

  return accuracy, mean_iou, mean_dice

In [ ]:
test_acc, test_iou, test_dice = evaluate(model, test_loader, device)

print(f"Test Accuracy: {test_acc:.2f}")
print(f"Mean IoU: {test_iou:.2f}")
print(f"Mean Dice: {test_dice:.2f}")

Test Accuracy: 0.43
Mean IoU: 0.16
Mean Dice: 0.24
